# 🎬 Classification Binaire de Texte avec IMDB

## 🎯 Objectifs d'Apprentissage

Ce challenge vous permettra de :
- Prétraiter des données textuelles pour les réseaux de neurones
- Construire et entraîner un réseau de neurones feedforward pour la classification binaire
- Évaluer les performances d'un modèle avec des données de validation et de test
- Visualiser les métriques d'entraînement et de validation pour détecter le surapprentissage

## 📊 Ce que vous allez créer

- Un modèle de classification binaire de texte utilisant le dataset IMDB pour classifier les critiques de films comme positives ou négatives
- Une visualisation de la perte et de la précision d'entraînement et de validation pour analyser les performances du modèle

---

**Note:** Ce notebook implémente toutes les étapes du challenge de manière complète et structurée.

In [ ]:
# === Partie 1: Prétraitement des données ===

# Imports nécessaires
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Dense
from keras import optimizers

# Configuration
num_words = 10000  # Garder les 10k mots les plus fréquents

# Charger le dataset IMDB depuis Keras
print("📥 Chargement du dataset IMDB...")
(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(num_words=num_words)

print(f"✓ Données d'entraînement: {len(train_data)} critiques")
print(f"✓ Données de test: {len(test_data)} critiques")
print(f"Exemple de critique (encodée): {train_data[0][:10]}...")
print(f"Label: {train_labels[0]} (1=positif, 0=négatif)")

# Fonction pour vectoriser les séquences (one-hot encoding)
def vectorize_sequences(sequences, dimension=10000):
    """
    Convertit les séquences d'entiers en matrices binaires.

    Chaque critique est transformée en un vecteur de 10,000 dimensions
    avec des 1 aux indices correspondant aux mots présents.
    """
    # Créer une matrice de zéros de forme (len(sequences), dimension)
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1  # Mettre à 1 les indices spécifiques
    return results

# Vectoriser les données d'entraînement et de test
print("\n🔄 Vectorisation des séquences...")
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# Convertir les labels en arrays numpy de type float
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

# Créer un ensemble de validation (10,000 échantillons)
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print(f"✓ Forme des données d'entraînement: {partial_x_train.shape}")
print(f"✓ Forme des données de validation: {x_val.shape}")
print(f"✓ Forme des données de test: {x_test.shape}")


# === Partie 2: Construction du modèle ===

print("\n🏗️  Construction du modèle feedforward...")

model = Sequential([
    # Première couche dense avec activation ReLU
    Dense(16, activation='relu', input_shape=(10000,)),
    # Deuxième couche dense avec activation ReLU
    Dense(16, activation='relu'),
    # Couche de sortie avec activation sigmoid pour classification binaire
    Dense(1, activation='sigmoid')
])

# Compilation du modèle
# - RMSprop optimizer: adaptatif, bon pour les réseaux feedforward
# - binary_crossentropy: fonction de perte pour classification binaire
# - accuracy: métrique d'évaluation
model.compile(
    optimizer=optimizers.RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✓ Modèle compilé avec succès")
print("\nArchitecture du modèle:")
model.summary()


# === Partie 3: Entraînement du modèle ===

print("\n🎓 Entraînement du modèle (20 epochs, batch_size=512)...")

history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

print("\n✓ Entraînement terminé")


# === Partie 4: Évaluation et visualisation ===

print("\n📊 Visualisation des métriques d'entraînement et de validation...")

# Récupérer l'historique des métriques
history_dict = history.history
loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']
acc_values = history_dict['accuracy']
val_acc_values = history_dict['val_accuracy']
epochs = range(1, len(loss_values) + 1)

# Créer les graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1: Perte (Loss)
ax1.plot(epochs, loss_values, 'bo-', label='Perte d'entraînement')
ax1.plot(epochs, val_loss_values, 'ro-', label='Perte de validation')
ax1.set_title('Perte d'entraînement vs validation', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Perte (Loss)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graphique 2: Précision (Accuracy)
ax2.plot(epochs, acc_values, 'bo-', label='Précision d'entraînement')
ax2.plot(epochs, val_acc_values, 'ro-', label='Précision de validation')
ax2.set_title('Précision d'entraînement vs validation', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Précision (Accuracy)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Détection du surapprentissage
# Le modèle commence à surappren dre quand la perte de validation commence à augmenter
# tandis que la perte d'entraînement continue de diminuer
min_val_loss_epoch = np.argmin(val_loss_values) + 1
print(f"\n🔍 Analyse du surapprentissage:")
print(f"   • La perte de validation minimale est atteinte à l'epoch {min_val_loss_epoch}")
print(f"   • Perte de validation minimale: {min(val_loss_values):.4f}")
print(f"   • Précision de validation correspondante: {val_acc_values[min_val_loss_epoch-1]:.4f}")

if min_val_loss_epoch < len(epochs):
    print(f"   ⚠️  Le modèle commence à surapprentir après l'epoch {min_val_loss_epoch}")
    print(f"   💡 Recommandation: Ré-entraîner avec {min_val_loss_epoch} epochs")


# === Partie 5: Ré-entraînement avec nombre optimal d'epochs ===

print(f"\n🔄 Ré-entraînement du modèle avec {min_val_loss_epoch} epochs...")

# Reconstruire un nouveau modèle
model_final = Sequential([
    Dense(16, activation='relu', input_shape=(10000,)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_final.compile(
    optimizer=optimizers.RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Entraîner sur TOUTES les données d'entraînement (train + validation)
# avec le nombre optimal d'epochs
history_final = model_final.fit(
    x_train,  # Toutes les données d'entraînement
    y_train,
    epochs=min_val_loss_epoch,
    batch_size=512,
    verbose=0
)

# Évaluer sur l'ensemble de test
test_loss, test_acc = model_final.evaluate(x_test, y_test, verbose=0)

print(f"\n📈 Résultats finaux sur l'ensemble de test:")
print(f"   • Perte (Loss): {test_loss:.4f}")
print(f"   • Précision (Accuracy): {test_acc:.4f} ({test_acc*100:.2f}%)")

# Comparaison avec le modèle initial (20 epochs)
initial_test_loss, initial_test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n📊 Comparaison avec le modèle initial (20 epochs):")
print(f"   • Précision initiale: {initial_test_acc:.4f} ({initial_test_acc*100:.2f}%)")
print(f"   • Précision optimisée: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   • Amélioration: {(test_acc - initial_test_acc)*100:+.2f}%")

# Analyse finale
print(f"\n✅ CHALLENGE TERMINÉ!")
print(f"   • Modèle de classification binaire construit et entraîné")
print(f"   • Précision finale sur le test: {test_acc*100:.2f}%")
print(f"   • Surapprentissage détecté et corrigé")
print(f"   • Visualisations des métriques créées")